# Reproducing the manuscript figures

This notebook regenerates the figures from *Technoeconomic Analysis of
Microgrids for AI Data Centers in the Continental United States* from the
released model and results.

The figures come in two tiers:

**Tier A — reproduced from the shipped results.** The continental-US maps and
distribution plots are generated directly from the harvested optimization
results in `output_tables/lcoe_results.csv`. That file is the
output of a multi-day CONUS optimization sweep and *cannot* be regenerated in a
notebook, so it ships with the repository. These cells need no API key and run
in seconds.

**Tier B — live, location-configurable examples.** A few figures illustrate
per-site behaviour (natural-gas LCOE scaling, hourly dispatch, state-of-charge,
islanded-share sweep). These run the model live for one example location and so
require an **NLR API key** to fetch that location's weather (see the key-check
cell before that section). They reproduce the *analogous* published figure and
can be pointed at any location; the published versions used specific sites noted
in each cell.

**Dependencies beyond the model's runtime requirements:** the maps need the geo
stack `geopandas`, `cartopy`, `h3`, and `shapely`. Install those in addition to
the packages listed in the README.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import h3
from shapely.geometry import Polygon


# Locate the repository root by walking up from the working directory until the
# marker files are found, so the notebook works whether it is launched from the
# repo root or from this notebooks/ folder. The model's src modules open the PUE
# lookup tables, hourly load shape, and fade surrogate via paths relative to the
# current working directory, so we chdir to the root.
def _find_repo_root() -> Path:
    markers = ("src/config.py", "output_tables/fade_surrogate.pkl")
    for start in (Path.cwd(), Path.cwd().parent):
        for d in (start, *start.parents):
            if all((d / m).exists() for m in markers):
                return d
    raise RuntimeError(
        "Could not locate the repository root. Run this notebook from within the "
        "cloned repository (markers: src/config.py, output_tables/fade_surrogate.pkl)."
    )


REPO = _find_repo_root()
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

# Prime the offline reverse-geocoder (used by the model's get_state_from_coords)
# in single-threaded mode. reverse_geocoder defaults to a multiprocessing loader
# that fails to spawn cleanly under some Windows notebook kernels; building the
# singleton single-threaded here makes every later lookup reuse this instance.
import reverse_geocoder as _rg
_rg.search([(39.0, -96.0)], mode=1)

DATA = REPO / "output_tables" / "lcoe_results.csv"

# Generated figures are written here when SAVE_FIGS is True; otherwise they are
# only displayed inline. The directory is git-ignored.
SAVE_FIGS = False
OUTDIR = REPO / "notebooks" / "figure_output"
OUTDIR.mkdir(parents=True, exist_ok=True)

GPU_COUNT = 100_000     # headline facility scale (MV-coupled regime)
DPI = 200

# Winner-overlay hatch styling (kept light so the LCOE fill stays readable).
HATCH_NG, HATCH_GRID = "..", "///"
HATCH_ALPHA, HATCH_LW = 0.30, 0.30


def _save(fig, name):
    """Save a figure to OUTDIR only when SAVE_FIGS is enabled."""
    if SAVE_FIGS:
        fig.savefig(OUTDIR / name, dpi=DPI, bbox_inches="tight", facecolor="white")
        print("saved:", name)


print("repo root:", REPO)

## Load the harvested results

The headline case is **100,000 GPUs, MV-coupled** (every 100k-GPU hex resolves
to the medium-voltage spine topology). The 24,000-GPU slice is used for one
low-voltage comparison figure.

In [ ]:
df = pd.read_csv(DATA)
mv = df[df["gpu_count"] == GPU_COUNT].copy()
assert (mv["dc_topology"] == "mv_coupled").all(), "expected all mv_coupled at 100k"
print(f"MV / {GPU_COUNT:,} GPU hexes: {len(mv)}")
print("median total LCOE ($/kWh):")
for c in ["dc_coupled_total_lcoe", "natural_gas_total_lcoe",
          "natural_gas_threeyr_total_lcoe", "grid_total_lcoe"]:
    print(f"  {c:34s} {mv[c].median():.4f}")

## Speed-premium economics

The speed-adjusted LCOE adds the discounted opportunity cost of GPU-hours lost
during construction to a system's energy LCOE; the manuscript defines it in
full. The shipped `*_total_lcoe` columns already include this adder. The helpers
below compute the adder on demand, so the GPU-indifference and
construction-breakeven figures can solve for it at arbitrary build times.

Parameters (`src/config.py`): 7% discount rate, 27-year horizon, $2.40/GPU-hr
opportunity cost, energy delivered at 99% uptime. Build times: DC solar 2.0 yr;
natural gas ~4 yr (`ng_construction_years`) plus a 3.0-yr accelerated case; grid
from per-state interconnection times.

In [ ]:
from lcoe_calc import GRID_BASELINE_DATA, GRID_DEFAULT, get_state_from_coords

DISCOUNT_RATE = 0.07
HOURS_PER_YEAR = 8760
EVALUATION_YEARS = 27
GPU_HOUR_SPOT_PRICE = 2.40
UPTIME_PCT = 99.0
SOLAR_CONSTRUCTION_YEARS = 2.0


def idling_npv(construction_years, gpu_count=GPU_COUNT,
               price=GPU_HOUR_SPOT_PRICE, r=DISCOUNT_RATE):
    """NPV of GPU idling opportunity cost over the construction window."""
    annual = gpu_count * price * HOURS_PER_YEAR
    full = int(construction_years)
    npv = sum(annual / (1 + r) ** y for y in range(full))
    frac = construction_years - full
    if frac > 0:
        npv += annual * frac / (1 + r) ** full
    return npv


def energy_npv(annual_energy_mwh, construction_years, uptime_pct=UPTIME_PCT,
               r=DISCOUNT_RATE, eval_years=EVALUATION_YEARS):
    """NPV of delivered energy (mid-year discounting), ops start at completion."""
    start = int(construction_years)
    first_frac = 1.0 - (construction_years - start)
    served = annual_energy_mwh * uptime_pct / 100.0
    flows = [0.0] * eval_years
    if start < eval_years:
        flows[start] = served * first_frac
    for y in range(start + 1, eval_years):
        flows[y] = served
    return sum(flows[y] / (1 + r) ** (y + 0.5) for y in range(eval_years))


def speed_premium(annual_energy_mwh, construction_years,
                  gpu_count=GPU_COUNT, price=GPU_HOUR_SPOT_PRICE):
    """$/kWh idling adder for a given build time and delivered-energy stream."""
    return (idling_npv(construction_years, gpu_count, price)
            / energy_npv(annual_energy_mwh, construction_years) / 1000.0)


# Per-hex interconnection years via the state of each hex. The state is derived
# live from coordinates with the model's offline reverse-geocoder, then mapped to
# the per-state interconnection time in GRID_BASELINE_DATA.
mv["resolved_state"] = [get_state_from_coords(lat, lon)
                        for lat, lon in zip(mv["latitude"], mv["longitude"])]
mv["interconnect_years"] = mv["resolved_state"].map(
    lambda s: GRID_BASELINE_DATA.get(s, GRID_DEFAULT)[1])
assert mv["interconnect_years"].notna().all(), "missing interconnect years"

_ae = mv["annual_energy_mwh"].to_numpy()

## Shared plotting helpers and color scale

Both hero maps use one shared color scale so they are directly comparable.
Bounds are the 0.5th/95th percentiles of the pooled best-of-three (winning)
total LCOE across the standard- and accelerated-natural-gas cases.

In [ ]:
def colorflip_viridis(level: float = 0.15) -> mcolors.Colormap:
    """Reversed viridis (yellow = low LCOE, purple = high), blended toward white."""
    arr = plt.colormaps["viridis_r"](np.linspace(0, 1, 256))
    arr[:, :3] = arr[:, :3] * (1 - level) + level
    return mcolors.LinearSegmentedColormap.from_list("viridis_flip_light", arr)


def h3_to_polygon(h3_index: str) -> Polygon:
    boundary = h3.cell_to_boundary(h3_index)          # (lat, lng) tuples in h3 v4
    return Polygon([(lng, lat) for lat, lng in boundary])


def winner_index(d, dc_col, ng_col, grid_col) -> np.ndarray:
    """0 = DC solar, 1 = natural gas, 2 = grid, by minimum total LCOE."""
    cost = np.column_stack([d[dc_col].to_numpy(), d[ng_col].to_numpy(), d[grid_col].to_numpy()])
    return cost.argmin(axis=1)


def best_of_three(d, dc_col, ng_col, grid_col):
    return np.minimum.reduce([d[dc_col].to_numpy(), d[ng_col].to_numpy(), d[grid_col].to_numpy()])


def add_conus_basemap(ax):
    """National/state borders and CONUS extent for an H3 hexmap axis."""
    ax.set_extent([-125, -66, 24, 50], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.6, zorder=10)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.6, zorder=10)
    ax.add_feature(cfeature.STATES.with_scale("50m"), edgecolor="white",
                   facecolor="none", linewidth=0.6, zorder=11)
    ax.add_feature(cfeature.STATES.with_scale("50m"), edgecolor="black",
                   facecolor="none", linewidth=0.3, zorder=12)


CONUS_PROJ = ccrs.LambertConformal(central_longitude=-96, central_latitude=39)
CMAP = colorflip_viridis(0.35)

_pooled = np.concatenate([
    best_of_three(mv, "dc_coupled_total_lcoe", "natural_gas_total_lcoe", "grid_total_lcoe"),
    best_of_three(mv, "dc_coupled_total_lcoe", "natural_gas_threeyr_total_lcoe", "grid_total_lcoe"),
])
LCOE_VMIN, LCOE_VMAX = float(np.quantile(_pooled, 0.005)), float(np.quantile(_pooled, 0.95))
print(f"shared colorbar (pooled p0.5/p95): {LCOE_VMIN:.4f} - {LCOE_VMAX:.4f}")

## Hero maps — three-way winner comparison

CONUS H3 hex map. Fill color is the winning (minimum) total LCOE among
{DC solar, natural gas, grid}; the winning technology is shown by hatch overlay
(DC solar = none, natural gas = dots, grid = diagonal lines). The two panels
differ only in the natural-gas build-time assumption.

In [ ]:
def make_3way_map(data, ng_col, ng_label, title, save_name):
    sub = data[["h3_index", "dc_coupled_total_lcoe", ng_col, "grid_total_lcoe"]].dropna().copy()
    wi = winner_index(sub, "dc_coupled_total_lcoe", ng_col, "grid_total_lcoe")
    best = np.column_stack([sub["dc_coupled_total_lcoe"], sub[ng_col],
                            sub["grid_total_lcoe"]]).min(axis=1)
    sub["winner"], sub["best_lcoe"] = wi, best

    counts = {lab: int((wi == i).sum())
              for i, lab in enumerate(["DC Solar", ng_label, "Grid"])}
    print(f"[{ng_label}] winner counts (n={len(sub)}): {counts}")

    sub["geometry"] = sub["h3_index"].apply(h3_to_polygon)
    gdf = gpd.GeoDataFrame(sub, geometry="geometry", crs="EPSG:4326")

    fig = plt.figure(figsize=(16, 10))
    ax = fig.add_subplot(1, 1, 1, projection=CONUS_PROJ)
    norm = plt.Normalize(vmin=LCOE_VMIN, vmax=LCOE_VMAX)
    for _, row in gdf.iterrows():
        ax.add_geometries([row.geometry], crs=ccrs.PlateCarree(),
                          facecolor=CMAP(norm(row["best_lcoe"])), edgecolor="none", linewidth=0)
    for win_val, hatch in [(1, HATCH_NG), (2, HATCH_GRID)]:
        wsub = gdf[gdf["winner"] == win_val]
        if len(wsub):
            ax.add_geometries(wsub.geometry, crs=ccrs.PlateCarree(), facecolor="none",
                              edgecolor="black", linewidth=HATCH_LW, hatch=hatch, alpha=HATCH_ALPHA)
    add_conus_basemap(ax)

    sm = plt.cm.ScalarMappable(cmap=CMAP, norm=norm); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.7, pad=0.02, aspect=30)
    cbar.set_label("LCOE ($/kWh)", fontsize=13)

    handles = [
        mpatches.Patch(facecolor=CMAP(0.15), edgecolor="black", linewidth=0.5, label="DC Solar"),
        mpatches.Patch(facecolor="lightgray", edgecolor="black", linewidth=0.5,
                       hatch=HATCH_NG * 3, label=ng_label),
        mpatches.Patch(facecolor="lightgray", edgecolor="black", linewidth=0.5,
                       hatch=HATCH_GRID, label="Grid"),
    ]
    leg = ax.legend(handles=handles, loc="lower right", frameon=True, fancybox=True,
                    framealpha=1.0, fontsize=11, title="Lower Cost Technology", title_fontsize=12)
    leg.set_zorder(100)
    ax.set_title(title, fontsize=15, fontweight="bold", pad=18)
    _save(fig, save_name)
    return fig


make_3way_map(
    mv, "natural_gas_total_lcoe", "Natural Gas",
    "Cost Comparison: DC Solar vs Natural Gas vs Grid\n"
    "100,000 GPU, 99% uptime Datacenter | Speed-premium LCOE",
    "comparison_3way_100k_std.png")
plt.show()

In [ ]:
make_3way_map(
    mv, "natural_gas_threeyr_total_lcoe", "Accelerated Natural Gas",
    "Cost Comparison: DC Solar vs Accelerated Natural Gas vs Grid\n"
    "100,000 GPU, 99% uptime Datacenter | Speed-premium LCOE",
    "comparison_3way_100k_3yr.png")
plt.show()

## Baseline DC-solar LCOE map

Energy-only (no speed premium) DC-coupled solar+storage LCOE across CONUS.
Yellow = cheap (high-insolation Southwest), purple = expensive (cloudy /
high-latitude).

In [ ]:
def make_value_hexmap(values, h3_idx, *, label, title, save_name, cmap=None, vmin=None, vmax=None):
    """Fill-only CONUS H3 hexmap of a single per-hex value."""
    sub = pd.DataFrame({"h3_index": h3_idx.to_numpy(), "val": values.to_numpy()}).dropna()
    sub["geometry"] = sub["h3_index"].apply(h3_to_polygon)
    gdf = gpd.GeoDataFrame(sub, geometry="geometry", crs="EPSG:4326")
    cmap = cmap if cmap is not None else CMAP
    vmin = float(sub["val"].quantile(0.05)) if vmin is None else vmin
    vmax = float(sub["val"].quantile(0.95)) if vmax is None else vmax
    norm = plt.Normalize(vmin=vmin, vmax=vmax)

    fig = plt.figure(figsize=(16, 10))
    ax = fig.add_subplot(1, 1, 1, projection=CONUS_PROJ)
    for _, row in gdf.iterrows():
        ax.add_geometries([row.geometry], crs=ccrs.PlateCarree(),
                          facecolor=cmap(norm(row["val"])), edgecolor="none", linewidth=0)
    add_conus_basemap(ax)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.7, pad=0.02, aspect=30)
    cbar.set_label(label, fontsize=13)
    ax.set_title(title, fontsize=15, fontweight="bold", pad=18)
    _save(fig, save_name)
    return fig


make_value_hexmap(
    mv["dc_coupled_base_lcoe"], mv["h3_index"],
    label="DC Solar LCOE ($/kWh)",
    title="Baseline DC Solar+Storage LCOE\n100,000 GPU, 99% uptime Datacenter | Energy-only LCOE",
    save_name="dc_solar_base_lcoe_100k.png")
plt.show()

## Price difference: natural gas vs DC solar — 24,000 GPU (LV)

Per-hex baseline (energy-only) LCOE difference, natural gas minus DC solar.
Blue = natural gas cheaper, orange = DC solar cheaper, centred on zero. Built at
**24,000 GPU** (the low-voltage topology) on purpose: the small-facility case,
where gas's scale economics are weakest, is still where it wins almost
everywhere on energy-only cost.

In [ ]:
GPU_DIFF = 24_000
diff_df = df[df["gpu_count"] == GPU_DIFF].copy()
# price_diff = NG base - DC base. Positive => DC solar cheaper (orange).
diff_df["price_diff"] = diff_df["natural_gas_base_lcoe"] - diff_df["dc_coupled_base_lcoe"]
n_dc = int((diff_df["price_diff"] > 0).sum())
n_ng = int((diff_df["price_diff"] < 0).sum())
mean_signed = float(diff_df["price_diff"].mean())
cheaper = "DC Solar" if mean_signed > 0 else "Natural Gas"
print(f"{GPU_DIFF} GPU baseline diff: DC cheaper {n_dc} hexes, NG cheaper {n_ng} hexes; "
      f"mean ${abs(mean_signed):.4f}/kWh ({cheaper} cheaper on avg)")

diff_cmap = mcolors.LinearSegmentedColormap.from_list(
    "ng_dc_diff", ["#1b6ca8", "#5fa8d3", "#ffffff", "#f2a154", "#c75413"], N=256)
DIFF_LIM = 0.40
norm = mcolors.TwoSlopeNorm(vmin=-DIFF_LIM, vcenter=0.0, vmax=DIFF_LIM)

sub = diff_df[["h3_index", "price_diff"]].dropna().copy()
sub["geometry"] = sub["h3_index"].apply(h3_to_polygon)
gdf = gpd.GeoDataFrame(sub, geometry="geometry", crs="EPSG:4326")

fig = plt.figure(figsize=(18, 10))
ax = fig.add_subplot(1, 1, 1, projection=CONUS_PROJ)
for _, row in gdf.iterrows():
    ax.add_geometries([row.geometry], crs=ccrs.PlateCarree(),
                      facecolor=diff_cmap(norm(row["price_diff"])), edgecolor="none", linewidth=0)
add_conus_basemap(ax)
sm = plt.cm.ScalarMappable(cmap=diff_cmap, norm=norm); sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.85, pad=0.02, aspect=30)
cbar.set_label("← Natural Gas Cheaper   |   DC Solar Cheaper →\n"
               "Price Difference ($/kWh)", fontsize=12)
ax.set_title("Price Difference: Natural Gas vs DC Solar\n"
             f"{GPU_DIFF:,} GPU, 99% uptime Datacenter | Baseline LCOE\n"
             f"Natural Gas: {n_ng} hexes, DC Solar: {n_dc} hexes",
             fontsize=14, fontweight="bold", pad=16)
_save(fig, "price_diff_ng_vs_dc_24k.png")
plt.show()

## AC-vs-DC coupling advantage

DC-coupled is cheaper than AC-coupled by construction (fewer conversion stages).
At 100k/MV the advantage is strictly positive in every hex.

In [ ]:
adv = mv["ac_coupled_base_lcoe"] - mv["dc_coupled_base_lcoe"]
adv_pct = adv / mv["ac_coupled_base_lcoe"] * 100.0
assert (adv > 0).all(), "unexpected AC<DC hexes"
print(f"DC advantage: abs ${adv.mean():.4f}/kWh mean ({adv.min():.4f}-{adv.max():.4f}); "
      f"pct {adv_pct.mean():.2f}% mean")

HIST_COLOR, MEAN_COLOR = "#5B9BD5", "#C55A11"


def advantage_hist(values, mean_fmt, xlabel, title, save_name):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(values, bins=40, alpha=0.8, edgecolor="black", linewidth=0.5, color=HIST_COLOR)
    m = values.mean()
    ax.axvline(m, color=MEAN_COLOR, linestyle="--", linewidth=2.5, label=f"Mean: {mean_fmt.format(m)}")
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel("Number of Locations", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, linestyle=":")
    fig.tight_layout()
    _save(fig, save_name)
    return fig


advantage_hist(adv, "${:.4f}/kWh", "DC Cost Advantage over AC ($/kWh)",
               "DC-Coupled Cost Advantage Distribution\n100,000 GPU (MV), 99% Uptime",
               "dc_advantage_absolute_100k.png")
plt.show()

advantage_hist(adv_pct, "{:.1f}%", "DC Cost Advantage over AC (%)",
               "DC-Coupled Percentage Cost Advantage\n100,000 GPU (MV), 99% Uptime",
               "dc_advantage_percentage_100k.png")
plt.show()

## GPU indifference price — DC solar vs grid

DC solar builds fast (2 yr) but has a higher energy base LCOE; grid is cheap
per-kWh but interconnects slowly. The crossover is the GPU spot price at which
the two have equal *total* LCOE — how much idle-GPU opportunity cost it takes
for DC's speed to pay for itself. Because the idling premium is linear in GPU
price, this has a closed form:

`p* = (grid_base - dc_base) / (f_dc - f_grid)`, with
`f = idling_npv(years, $1/hr) / energy_npv`.

`p* < 0` means DC is cheaper even at a $0 GPU price (strict dominance). Hexes
are colored by `p*`, centred on the current $2.40 index; grey hatch = DC
strictly dominates.

In [ ]:
def _idle_factor_per_dollar(years):
    """Idling premium ($/kWh) per $1/GPU-hr, over each hex's annual energy."""
    return np.array([idling_npv(years, price=1.0) / energy_npv(a, years) / 1000.0 for a in _ae])


_f_dc = _idle_factor_per_dollar(SOLAR_CONSTRUCTION_YEARS)
_f_grid = np.array([idling_npv(y, price=1.0) / energy_npv(a, y) / 1000.0
                    for a, y in zip(_ae, mv["interconnect_years"].to_numpy())])

gpu_indiff = ((mv["grid_base_lcoe"].to_numpy() - mv["dc_coupled_base_lcoe"].to_numpy())
              / (_f_dc - _f_grid))
mv["gpu_indiff_price"] = gpu_indiff
dc_dominates = gpu_indiff < 0

print(f"GPU indifference p* (DC vs grid), n={len(mv)}")
print(f"  DC strictly dominates (p*<0): {int(dc_dominates.sum())} ({100 * dc_dominates.mean():.1f}%)")
_comp = gpu_indiff[~dc_dominates]
print(f"  competitive p* median ${np.median(_comp):.2f}/hr; "
      f"p05-p95 ${np.quantile(_comp, 0.05):.2f}-${np.quantile(_comp, 0.95):.2f}")
print(f"  current index ${GPU_HOUR_SPOT_PRICE:.2f}/hr -> "
      f"{int((_comp < GPU_HOUR_SPOT_PRICE).sum())} competitive hexes favor grid")

In [ ]:
def make_gpu_indifference_map(save_name):
    sub = mv[["h3_index", "gpu_indiff_price"]].dropna().copy()
    sub["dc_dominates"] = sub["gpu_indiff_price"] < 0
    sub["geometry"] = sub["h3_index"].apply(h3_to_polygon)
    gdf = gpd.GeoDataFrame(sub, geometry="geometry", crs="EPSG:4326")

    colors = ["#0072B2", "#56B4E9", "#FFFFFF", "#E69F00", "#D55E00"]
    cmap = mcolors.LinearSegmentedColormap.from_list("cool_to_hot", colors, N=256)
    comp = gdf.loc[~gdf["dc_dominates"], "gpu_indiff_price"].to_numpy()
    half = max(abs(np.percentile(comp, 5) - GPU_HOUR_SPOT_PRICE),
               abs(np.percentile(comp, 95) - GPU_HOUR_SPOT_PRICE)) if len(comp) else 1.5
    norm = plt.Normalize(vmin=GPU_HOUR_SPOT_PRICE - half, vmax=GPU_HOUR_SPOT_PRICE + half)

    fig = plt.figure(figsize=(16, 10))
    ax = fig.add_subplot(1, 1, 1, projection=CONUS_PROJ)
    for _, row in gdf[~gdf["dc_dominates"]].iterrows():
        ax.add_geometries([row.geometry], crs=ccrs.PlateCarree(),
                          facecolor=cmap(norm(row["gpu_indiff_price"])), edgecolor="none", linewidth=0)
    dom_gdf = gdf[gdf["dc_dominates"]]
    if len(dom_gdf):
        ax.add_geometries(dom_gdf.geometry, crs=ccrs.PlateCarree(), facecolor="#808080",
                          edgecolor="black", linewidth=0.3, hatch="///", alpha=0.7)
    add_conus_basemap(ax)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.7, pad=0.02, aspect=25)
    cbar.set_label("GPU Indifference Price ($/hr)", fontsize=13)
    cbar.ax.yaxis.set_major_formatter(plt.FormatStrFormatter("$%.2f"))

    handles = [
        mpatches.Patch(facecolor="#0072B2", label="< index (grid favored)"),
        mpatches.Patch(facecolor="#FFFFFF", edgecolor="black", linewidth=0.5,
                       label=f"index (${GPU_HOUR_SPOT_PRICE:.2f}/hr)"),
        mpatches.Patch(facecolor="#D55E00", label="> index (DC favored)"),
        mpatches.Patch(facecolor="#808080", edgecolor="black", hatch="///",
                       label="DC strictly dominates"),
    ]
    leg = ax.legend(handles=handles, loc="lower right", frameon=True, fancybox=True,
                    framealpha=1.0, fontsize=11, title="GPU price vs DC island", title_fontsize=12)
    leg.set_zorder(100)
    ax.set_title("GPU Indifference Price: DC Solar vs Grid\n"
                 "100,000 GPU, 99% uptime Datacenter | white = current index price",
                 fontsize=15, fontweight="bold", pad=18)
    _save(fig, save_name)
    return fig


make_gpu_indifference_map("gpu_indifference_dc_vs_grid_100k.png")
plt.show()

## Grid interconnection breakeven — required timeline to match DC

How fast would the grid have to interconnect for grid *total* LCOE to equal DC
solar's? Solve `grid_base + premium(interconnect_years) = dc_total` for the
years. Lower (purple) = grid must be very fast to compete; higher (yellow) =
grid can be slow and still win.

In [ ]:
from scipy.optimize import brentq


def _grid_breakeven_years(row):
    dc_total, gbase, ae = row["dc_coupled_total_lcoe"], row["grid_base_lcoe"], row["annual_energy_mwh"]
    if gbase >= dc_total:          # grid base alone already exceeds DC total
        return np.nan
    try:
        return brentq(lambda yy: gbase + speed_premium(ae, yy) - dc_total, 0.01, 15.0)
    except ValueError:
        return np.nan


mv["grid_breakeven_years"] = mv.apply(_grid_breakeven_years, axis=1)
_be = mv["grid_breakeven_years"].dropna()
print(f"interconnect breakeven years: valid {len(_be)}/{len(mv)}")
print(f"  median {_be.median():.2f} yr; p05-p95 {_be.quantile(0.05):.2f}-{_be.quantile(0.95):.2f}; "
      f"range {_be.min():.2f}-{_be.max():.2f}")
print(f"  faster than current interconnect: "
      f"{int((mv['grid_breakeven_years'] < mv['interconnect_years']).sum())} hexes")

In [ ]:
def make_breakeven_map(value_col, label, title, save_name, vmin=2.0, vmax=5.0):
    """CONUS hexmap of a breakeven-years column (yellow = slow OK, purple = must be fast)."""
    sub = mv[["h3_index", value_col]].dropna().copy()
    sub["geometry"] = sub["h3_index"].apply(h3_to_polygon)
    gdf = gpd.GeoDataFrame(sub, geometry="geometry", crs="EPSG:4326")
    cmap = colorflip_viridis(0.15)
    norm = plt.Normalize(vmin=vmin, vmax=vmax)

    fig = plt.figure(figsize=(16, 10))
    ax = fig.add_subplot(1, 1, 1, projection=CONUS_PROJ)
    for _, row in gdf.iterrows():
        ax.add_geometries([row.geometry], crs=ccrs.PlateCarree(),
                          facecolor=cmap(norm(row[value_col])), edgecolor="none", linewidth=0)
    add_conus_basemap(ax)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.7, pad=0.02, aspect=30)
    cbar.set_label(label, fontsize=12, fontweight="bold")
    ax.set_title(title, fontsize=15, fontweight="bold", pad=18)
    _save(fig, save_name)
    return fig


make_breakeven_map(
    "grid_breakeven_years", "Required Grid Interconnection (years to match DC)",
    "Grid Interconnection Breakeven vs DC Solar\n"
    "100,000 GPU, 99% uptime Datacenter | Speed-premium LCOE",
    "grid_interconnect_breakeven_100k.png")
plt.show()

## Natural-gas construction breakeven — required build time to match DC

Mirror of the grid timeline: how fast would a gas plant have to be built for its
*total* LCOE to equal DC solar's? Hexes where the gas energy base alone already
exceeds DC's total (gas can never match, at any build speed) are dropped.

In [ ]:
def _ng_breakeven_years(row):
    dc_total, ng_base, ae = row["dc_coupled_total_lcoe"], row["natural_gas_base_lcoe"], row["annual_energy_mwh"]
    if ng_base >= dc_total:          # gas energy base alone already exceeds DC total
        return np.nan
    try:
        return brentq(lambda yy: ng_base + speed_premium(ae, yy) - dc_total, 0.01, 15.0)
    except ValueError:
        return np.nan


mv["ng_breakeven_years"] = mv.apply(_ng_breakeven_years, axis=1)
_nbe = mv["ng_breakeven_years"].dropna()
_never = int(mv["ng_breakeven_years"].isna().sum())
print(f"NG construction breakeven years: valid {len(_nbe)}/{len(mv)} "
      f"({_never} hexes gas cannot match DC at any build speed)")
print(f"  median {_nbe.median():.2f} yr; p05-p95 {_nbe.quantile(0.05):.2f}-{_nbe.quantile(0.95):.2f}; "
      f"range {_nbe.min():.2f}-{_nbe.max():.2f}")
print(f"  must beat the standard 4.0-yr build to win: "
      f"{int((mv['ng_breakeven_years'] < mv['ng_construction_years']).sum())} hexes")

make_breakeven_map(
    "ng_breakeven_years", "Required Gas Build Time (years to match DC)",
    "Natural Gas Construction Breakeven vs DC Solar\n"
    "100,000 GPU, 99% uptime Datacenter | Speed-premium LCOE",
    "ng_construction_breakeven_100k.png")
plt.show()

# Live, location-configurable figures

The cells below run the model live for one example location and need an **NLR
API key** to fetch that location's weather (NSRDB PSM4 TMY). Request a free key
at <https://developer.nlr.gov/signup/> and set two environment variables before
launching the notebook:

```bash
export NLR_API_KEY=your_key
export NLR_EMAIL=you@example.com
```

Weather for each location is cached under `output_tables/nsrdb_cache/` after the
first fetch, so re-runs are fast. These figures reproduce the *analogous*
published figure; the example sites match the published ones where noted, and
any location can be substituted.

In [ ]:
# Key check — the live cells below will fail to fetch weather without it.
_have_key = bool(os.environ.get("NLR_API_KEY")) and bool(os.environ.get("NLR_EMAIL"))
if not _have_key:
    print("NLR_API_KEY / NLR_EMAIL are not set. The live cells below need them to "
          "fetch weather for any location not already in output_tables/nsrdb_cache/. "
          "See the section header for setup instructions.")
else:
    print("NLR credentials detected; live weather fetches enabled.")

## Natural-gas LCOE scaling vs facility size

The CONUS sweep fixes facility size at 100k GPUs; this figure instead sweeps
size at a single representative site to show how the cost-optimal turbine
configuration (aeroderivative / F-class / H-class, simple vs combined cycle)
shifts with scale. Solar reference lines come from the nearest harvested hex.
Runs the model live for each size — expect a minute or two.

In [ ]:
import time
from datacenter_analyzer import DatacenterAnalyzer
from natgas_system_tool import (NGPowerPlantCalculator, generate_plant_configurations,
                                 TURBINE_LIBRARY)
from lcoe_calc import calculate_gas_system_lcoe
from config import load_config

# San Luis Valley, south-central Colorado — high-insolation hex where PV is cheap.
NG_SITE_LAT, NG_SITE_LON = 37.3886, -105.6743
NG_UPTIME = 99.0
_cfg = load_config()
_state = get_state_from_coords(NG_SITE_LAT, NG_SITE_LON)
_gas = (GRID_BASELINE_DATA.get(_state, GRID_DEFAULT)[3]
        if _state else _cfg.costs.default_gas_price_mmbtu)
_gas = _gas if _gas is not None else _cfg.costs.default_gas_price_mmbtu


def _gpu_sweep():
    counts = list(range(3000, 15000, 1200))
    counts += list(range(15000, 50000, 2400))
    counts += list(range(50000, 250000, 4800))
    return counts


print(f"NG scaling site: ({NG_SITE_LAT}, {NG_SITE_LON}) {_state}; "
      f"gas ${_gas:.2f}/MMBtu; uptime {NG_UPTIME}%")

In [ ]:
_rows = []
_t0 = time.time()
for _gpu in _gpu_sweep():
    _an = DatacenterAnalyzer(latitude=NG_SITE_LAT, longitude=NG_SITE_LON, total_gpus=_gpu)
    _fl = _an.calculate_facility_load(required_uptime_pct=NG_UPTIME)
    _ng = NGPowerPlantCalculator(facility_load=_fl, required_uptime_pct=NG_UPTIME,
                                 gas_price_mmbtu=_gas, efficiency_params=_cfg)
    _cfgs = generate_plant_configurations(
        _ng.required_generation_mw, TURBINE_LIBRARY, _fl.design_ambient_temp_c,
        require_n_minus_1=False, config=_cfg, annual_energy_mwh=_ng.annual_energy_mwh)
    if not _cfgs:
        continue
    _best, _best_lcoe = None, float("inf")
    for _pc in _cfgs:
        try:
            _r = calculate_gas_system_lcoe(plant_config=_pc, gas_price=_gas,
                                           facility_load=_fl, config=_cfg)
        except Exception:
            continue
        if _r.lcoe < _best_lcoe:
            _best_lcoe, _best = _r.lcoe, _pc
    if _best is not None:
        _rows.append({"gpus": _gpu, "facility_mw": _fl.facility_load_design_mw,
                      "ng_lcoe": _best_lcoe, "turbine_class": _best.turbine_class,
                      "cycle_type": _best.cycle_type, "n_units": _best.n_units})
ng_scale = pd.DataFrame(_rows)
print(f"swept {len(ng_scale)} sizes in {time.time() - _t0:.0f}s")

In [ ]:
# AC/DC solar reference LCOEs at the nearest harvested hex (scale-invariant within a topology).
_d100 = df[df["gpu_count"] == GPU_COUNT]
_near = ((_d100["latitude"] - NG_SITE_LAT) ** 2 + (_d100["longitude"] - NG_SITE_LON) ** 2).idxmin()
_ac_ref = float(_d100.loc[_near, "ac_coupled_base_lcoe"])
_dc_ref = float(_d100.loc[_near, "dc_coupled_base_lcoe"])

_CLASS_NAME = {"aero": "Aeroderivative", "f_class": "F-class", "h_class": "H-class"}
_CYCLE_NAME = {"SC": "Simple Cycle", "CC": "Combined Cycle"}
_MARK = {"SC": "o", "CC": "s"}
_COLOR = {"aero": "#FF6B6B", "f_class": "#4ECDC4", "h_class": "#45B7D1"}

fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(ng_scale["facility_mw"], ng_scale["ng_lcoe"], color="gray", alpha=0.3, lw=1, zorder=1)
for tc in ["aero", "f_class", "h_class"]:
    for cyc in ["SC", "CC"]:
        m = (ng_scale["turbine_class"] == tc) & (ng_scale["cycle_type"] == cyc)
        if m.any():
            ax.scatter(ng_scale.loc[m, "facility_mw"], ng_scale.loc[m, "ng_lcoe"],
                       color=_COLOR[tc], marker=_MARK[cyc], s=80, alpha=0.85,
                       edgecolors="black", linewidth=0.5,
                       label=f"{_CLASS_NAME[tc]} – {_CYCLE_NAME[cyc]}")
ax.axhline(_ac_ref, color="#E69F00", linestyle="--", lw=2, alpha=0.9,
           label=f"AC Solar+Storage (${_ac_ref:.3f})")
ax.axhline(_dc_ref, color="#2ca02c", linestyle="--", lw=2, alpha=0.9,
           label=f"DC Solar+Storage (${_dc_ref:.3f})")
ax.set_xlabel("Datacenter Nameplate Capacity (MW)", fontsize=14)
ax.set_ylabel("LCOE ($/kWh)", fontsize=14)
ax.set_title("Natural Gas LCOE Scaling Analysis\n"
             f"{_state}: ({NG_SITE_LAT:.2f}°, {NG_SITE_LON:.2f}°) | "
             f"Gas: ${_gas:.1f}/MMBtu | Uptime: {NG_UPTIME:.0f}%",
             fontsize=15, fontweight="bold")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right", fontsize=10, ncol=2, framealpha=0.95)
fig.tight_layout()
_save(fig, "ng_lcoe_scaling.png")
plt.show()

## Hourly dispatch and battery state-of-charge — example site

Optimizes a DC-coupled microgrid for one site and scale, then runs one
representative year of hourly dispatch. The first plot shows a representative
week (PV→load, battery→load, charging, curtailment against the IT + cooling
load); the second shows the annual state-of-charge distribution. Published SI
versions used Southern California (SOC) and a DC-coupled 100k facility
(dispatch); the example here is Houston, configurable below.

In [ ]:
from math import sqrt
from microgrid_optimizer import MicrogridOptimizer, SystemCosts
from pvstoragesim import evaluate_system, get_solar_generation

DISP_LAT, DISP_LON, DISP_NAME = 29.76, -95.37, "Houston, TX"
DISP_GPUS = 100_000
DISP_ARCH, DISP_TOPO = "dc_coupled", "mv_coupled"
DISP_UPTIME = 99.0

C_PV, C_BAT, C_CHG, C_CURT = "#E69F00", "#009E73", "#0072B2", "0.6"
C_COOL = "#9ECAE1"

_dcfg = load_config()
_analyzer = DatacenterAnalyzer(DISP_LAT, DISP_LON, DISP_GPUS, config=_dcfg)
DISP_FAC = _analyzer.calculate_facility_load(required_uptime_pct=DISP_UPTIME)

_costs = SystemCosts(
    solar_cost_per_kw=_dcfg.costs.solar_cost_y0,
    battery_cost_per_kw=_dcfg.costs.bess_cost_y0,
    solar_bos_cost_per_kw=_dcfg.costs.solar_bos_cost_y0_dc,
    battery_bos_cost_per_kw=_dcfg.costs.battery_bos_cost_y0_dc,
)
_opt = MicrogridOptimizer(
    latitude=DISP_LAT, longitude=DISP_LON, facility_load=DISP_FAC,
    required_uptime_pct=DISP_UPTIME, costs=_costs, architecture=DISP_ARCH,
    efficiency_params=_dcfg, topology=DISP_TOPO, verbose=False, seed=1)
_design = _opt.optimize()
DISP_SOLAR_MW = _design.solar_mw
DISP_BAT_MW = _design.battery_mw
DISP_DUR_H = _design.battery_mwh / _design.battery_mw
print(f"{DISP_NAME}: facility {DISP_FAC.facility_load_design_mw:.1f} MW | "
      f"design {DISP_SOLAR_MW:.1f} MW PV, {DISP_BAT_MW:.1f} MW / {_design.battery_mwh:.0f} MWh")

_solar_y0 = get_solar_generation(DISP_LAT, DISP_LON, facility_load=DISP_FAC)
disp_sim = evaluate_system(
    DISP_LAT, DISP_LON, solar_capacity_mw=DISP_SOLAR_MW, battery_power_mw=DISP_BAT_MW,
    battery_duration_hours=DISP_DUR_H, facility_load=DISP_FAC, architecture=DISP_ARCH,
    topology=DISP_TOPO, efficiency_params=_dcfg, solar_profile=_solar_y0, return_hourly=True)
print(f"year-0 uptime {disp_sim.uptime_pct:.2f}% | {disp_sim.battery_cycles_per_year:.0f} cycles/yr")

In [ ]:
def pick_cooling_week(hd):
    h = hd.copy()
    h["week"] = np.arange(len(h)) // 168
    full = h.groupby("week").size() == 168
    return int(h.groupby("week")["cooling_load_mw"].std()[full].idxmax())


def plot_dispatch(hd, name, week=None):
    """Two natural-scale panels: PV disposition, and meeting the IT+cooling load."""
    week = pick_cooling_week(hd) if week is None else week
    w = hd.iloc[week * 168:(week + 1) * 168].reset_index(drop=True)
    x = np.arange(len(w))
    sdc = w["solar_dc_mw"].to_numpy()
    curt = w["curtailed_solar_mw"].to_numpy()
    chg = w["battery_charge_mw"].to_numpy()
    bat = w["battery_discharge_mw"].to_numpy()
    it = w["it_load_mw"].to_numpy()
    total = w["total_load_mw"].to_numpy()
    unmet = w["unmet_load_mw"].to_numpy()
    pv_load_gross = np.maximum(sdc - chg - curt, 0.0)
    served = np.maximum(total - unmet, 0.0)
    bat_to_load = np.minimum(bat, served)
    pv_to_load = np.maximum(served - bat_to_load, 0.0)

    fig, (axA, axB) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    axA.fill_between(x, 0, pv_load_gross, color=C_PV, label="PV → load")
    axA.fill_between(x, pv_load_gross, pv_load_gross + chg, color=C_CHG, label="PV → battery (charge)")
    axA.fill_between(x, pv_load_gross + chg, pv_load_gross + chg + curt, color=C_CURT,
                     alpha=0.7, label="curtailed PV")
    axA.set_ylabel("PV generation (MW)")
    axA.set_title("A. Generation & disposition of PV", loc="left", fontweight="bold", fontsize=12)
    axA.legend(loc="upper right", ncol=3, framealpha=0.95, fontsize=9)
    axA.grid(axis="y", alpha=0.2, ls=":")

    axB.fill_between(x, 0, pv_to_load, color=C_PV, alpha=0.85, label="PV → load")
    axB.fill_between(x, pv_to_load, pv_to_load + bat_to_load, color=C_BAT, alpha=0.9, label="battery → load")
    axB.fill_between(x, it, total, color=C_COOL, alpha=0.45, label="cooling (IT→total)")
    axB.plot(x, it, color="black", lw=1.0, label="IT load")
    axB.plot(x, total, color="black", lw=1.7, label="total load")
    axB.set_ylim(0, total.max() * 1.3)
    axB.set_ylabel("load & supply (MW)")
    axB.set_title("B. Meeting the IT + cooling load", loc="left", fontweight="bold", fontsize=12)
    axB.legend(loc="upper right", ncol=3, framealpha=0.95, fontsize=9)
    axB.grid(axis="y", alpha=0.2, ls=":")
    for ax in (axA, axB):
        for d in range(8):
            ax.axvline(d * 24, color="gray", alpha=0.3, ls="--", lw=0.8)
    axB.set_xlim(0, 168); axB.set_xticks(np.arange(0, 169, 24))
    axB.set_xticklabels([f"day {i}" for i in range(8)])
    axB.set_xlabel("hour of representative week")
    fig.suptitle(f"Hourly dispatch — DC microgrid ({name}, {DISP_GPUS:,} GPUs, MV-coupled), week {week}",
                 fontweight="bold", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    _save(fig, "dispatch_example.png")
    return fig


plot_dispatch(disp_sim.hourly_data, DISP_NAME)
plt.show()

In [ ]:
def plot_soe(hd, name):
    cap = hd["battery_soc_mwh"].max()
    soe = hd["battery_soc_mwh"].to_numpy() / cap * 100
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.hist(soe, bins=50, alpha=0.8, color="#0072B2", edgecolor="white")
    ax.axvline(soe.mean(), color="#D55E00", ls="--", lw=2, label=f"Mean {soe.mean():.1f}%")
    ax.axvline(np.median(soe), color="#E69F00", ls="--", lw=2, label=f"Median {np.median(soe):.1f}%")
    ax.set_xlabel("State of Charge (% of total effective capacity)")
    ax.set_ylabel("Hours per year")
    ax.set_title(f"Annual Distribution of BESS State of Charge — {name}",
                 fontweight="bold", loc="left")
    ax.legend(); ax.grid(alpha=0.25, ls=":")
    fig.tight_layout()
    _save(fig, "battery_soe_distribution_example.png")
    return fig


plot_soe(disp_sim.hourly_data, DISP_NAME)
plt.show()

## Microgrid cost vs islanded share — reusable sweep

Sweeps the required uptime (islanded share of annual energy) at a single site
and compares DC-solar+storage against natural gas, with the local grid price as
a reference line. Each DC point re-optimizes the microgrid, so this is the
slowest cell — the demo below runs a **coarse** sweep at one location. Increase
`UPTIMES` (and change `SWEEP_LAT/LON`) for a denser or different reproduction;
the published figure used a fine grid across four reference locations, generated
offline.

In [ ]:
SWEEP_LAT, SWEEP_LON, SWEEP_NAME = 34.05, -117.75, "Southern California"
SWEEP_GPUS = 100_000
UPTIMES = [80.0, 90.0, 99.0]   # coarse demo; the published sweep was much finer


from lcoe_calc import compare_datacenter_power_systems


def uptime_sweep(lat, lon, gpu_count, uptimes, topology="mv_coupled"):
    """For each required uptime, run the full system comparison and return the
    DC-solar, natural-gas, and grid LCOEs. Uses the same public entry point as
    the CLI (compare_datacenter_power_systems); each point re-optimizes AC+DC
    solar, so this is the slowest cell in the notebook."""
    rows = []
    for up in uptimes:
        try:
            cmp = compare_datacenter_power_systems(
                total_gpus=gpu_count, required_uptime_pct=up,
                location=(lat, lon), topology=topology)
            rows.append({"uptime_pct": up, "facility_mw": cmp.facility_load_mw,
                         "dc_solar_lcoe": cmp.dc_solar.lcoe, "ng_lcoe": cmp.natural_gas.lcoe,
                         "grid_lcoe": cmp.grid_baseline.lcoe})
        except Exception as e:
            print(f"  uptime {up}: comparison failed ({e})")
    return pd.DataFrame(rows)


sweep = uptime_sweep(SWEEP_LAT, SWEEP_LON, SWEEP_GPUS, UPTIMES)
grid_ref = float(sweep["grid_lcoe"].median())
print(sweep.round(4).to_string(index=False))
print(f"grid reference: ${grid_ref:.4f}/kWh")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
ax.plot(sweep["uptime_pct"], sweep["dc_solar_lcoe"], "o-", color="#1f77b4", lw=3, label="DC Solar")
ax.plot(sweep["uptime_pct"], sweep["ng_lcoe"], "s-", color="#d9700a", lw=3, label="Natural Gas")
ax.axhline(grid_ref, color="black", ls="--", lw=2.5)
ax.text(0.985, grid_ref, f" Grid: ${grid_ref:.2f}/kWh", transform=ax.get_yaxis_transform(),
        ha="right", va="bottom", fontsize=12, fontweight="bold")
ax.set_xlabel("Islanded Share of Annual Energy / Required Uptime (%)", fontsize=13)
ax.set_ylabel("LCOE ($/kWh)", fontsize=13)
ax.set_title(f"Microgrid Energy Cost vs Annual Utilization\n"
             f"{SWEEP_NAME} ({sweep['facility_mw'].iloc[0]:.1f} MW Facility)",
             fontsize=15, fontweight="bold", loc="left")
ax.grid(True, alpha=0.3)
ax.legend(title="Microgrid Technology", fontsize=12, title_fontsize=13, loc="upper center", frameon=False)
fig.tight_layout()
_save(fig, "lcoe_vs_uptime_example.png")
plt.show()